# PI-Gold-Delta: Eventhouse → gold.fact_pi Incremental Load

Watermark-based incremental pipeline that reads new PI sensor events from the
`pi-realtime-db` Eventhouse (KQL `PiEvents` table) and appends them to
`gold.fact_pi` (Lakehouse Delta table).

**Schedule:** Every 15–30 minutes via Fabric pipeline or notebook scheduler.

**Flow:**
1. Read watermark (`MAX(Timestamp)` from `gold.fact_pi`)
2. Query `PiEvents` from Eventhouse for records after watermark
3. Filter bad-quality readings (`Questionable = false`)
4. Append to `gold.fact_pi` with dedup


In [ ]:
# Configuration
from pyspark.sql import functions as F
from datetime import datetime, timedelta

# Eventhouse connection
KUSTO_URI = "https://trd-8a08ckb2duw406mvvg.z2.kusto.fabric.microsoft.com"
KUSTO_DB = "pi-realtime-db"

# Target table
GOLD_TABLE = "gold.fact_pi"

# Safety: max rows per batch to prevent runaway loads
MAX_ROWS_PER_BATCH = 5_000_000

# Fallback watermark if gold.fact_pi is empty or doesn't exist
FALLBACK_WATERMARK = "2026-05-31 00:00:00"

def _kusto_tok():
    try:
        import notebookutils as _n; _c = _n.credentials
    except Exception:
        from notebookutils import mssparkutils as _m; _c = _m.credentials
    for _a in (KUSTO_URI, "kusto", "pbi"):
        try:
            _t = _c.getToken(_a)
            if _t: return _t
        except Exception:
            pass
    raise RuntimeError("could not acquire Kusto token")

def read_kusto(query):
    """Read from Eventhouse KQL database via Kusto Spark connector"""
    return (spark.read
        .format("com.microsoft.kusto.spark.datasource")
        .option("accessToken", _kusto_tok())
        .option("kustoCluster", KUSTO_URI)
        .option("kustoDatabase", KUSTO_DB)
        .option("kustoQuery", query)
        .load())

print(f"PI-Gold-Delta pipeline initialized")
print(f"  Eventhouse: {KUSTO_URI} / {KUSTO_DB}")
print(f"  Target: {GOLD_TABLE}")


## Step 1: Get Watermark
Read the latest timestamp already in `gold.fact_pi` to determine where to start loading.


In [ ]:
# Get watermark: latest timestamp already in gold.fact_pi
try:
    watermark_row = spark.sql(f"SELECT MAX(Timestamp) as max_ts FROM {GOLD_TABLE}").collect()[0]
    watermark = watermark_row['max_ts']
    if watermark is None:
        watermark = FALLBACK_WATERMARK
        print(f"Table exists but is empty — using fallback watermark: {watermark}")
    else:
        print(f"Watermark from {GOLD_TABLE}: {watermark}")
except Exception as e:
    watermark = FALLBACK_WATERMARK
    print(f"Table not found ({e}) — using fallback watermark: {watermark}")

# Query window: watermark → now
load_start = str(watermark)
load_end = datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')
print(f"\nLoad window: {load_start} → {load_end}")


## Step 2: Read New Events from Eventhouse
Query `PiEvents` KQL table for records after the watermark. Filter out questionable readings.


In [ ]:
# Read new PI events from Eventhouse
kql_query = f"""
    PiEvents
    | where Ts > datetime({load_start})
    | where Ts <= datetime({load_end})
    | project Tag, Timestamp = Ts,
             ValueNumeric = toreal(Value),
             ValueString = iif(isnull(toreal(Value)), tostring(Value), tostring('')),
             Good = not(Questionable), Questionable,
             IsSystem = false
    | order by Tag asc, Timestamp asc
"""

print(f"Querying Eventhouse for new events...")
raw_events = read_kusto(kql_query)
raw_count = raw_events.count()

print(f"✓ Retrieved {raw_count:,} raw PI events")

if raw_count == 0:
    print("No new data to load — exiting.")
    try:
        import notebookutils
        notebookutils.notebook.exit("No new data")
    except:
        pass
elif raw_count > MAX_ROWS_PER_BATCH:
    print(f"⚠️ {raw_count:,} rows exceeds batch limit of {MAX_ROWS_PER_BATCH:,}")
    print(f"   Loading first {MAX_ROWS_PER_BATCH:,} rows. Re-run to catch up.")
    raw_events = raw_events.limit(MAX_ROWS_PER_BATCH)

# Enrich with gold schema columns (matching rebuild-gold-fact-pi.py)
df_auth_pi = spark.table("gold.bridge_pi_tag_to_asset")
df_inf_pi = (spark.table("bridge_pi_tag_to_equipment")
    .select("Tag",
            F.col("Plant").alias("plant_inferred"),
            F.col("icare_id").alias("icare_id_inferred"),
            F.col("mapping_confidence").alias("pi_mapping_confidence"),
            F.col("mapping_jaccard").alias("pi_mapping_jaccard"))
)

new_events = (raw_events
    .join(df_auth_pi, "Tag", "left")
    .join(df_inf_pi, "Tag", "left")
    .withColumn("date_key", F.date_format("Timestamp", "yyyyMMdd").cast("int"))
    .withColumn("pi_match_source",
        F.when(F.col("asset_id").isNotNull(), F.lit("intake"))
         .when(F.col("icare_id_inferred").isNotNull(), F.lit("inferred"))
         .otherwise(F.lit("none")))
    .withColumn("icare_id", F.col("icare_id_inferred"))
    .withColumn("plant", F.coalesce(F.col("plant_inferred"),
        F.when(F.col("Tag").startswith("RV2:"), F.lit("RV2"))
         .when(F.col("Tag").startswith("RV3:"), F.lit("RV3"))))
    .select(
        "Tag", "asset_id", "plant", "Timestamp", "date_key",
        "ValueNumeric", "ValueString", "IsSystem",
        "Good", "Questionable",
        F.lit(None).cast("string").alias("WebId"),
        F.lit(None).cast("string").alias("DataServer"),
        "icare_id", "pi_mapping_confidence", "pi_mapping_jaccard",
        "tag_role", "downtime_relevance", "pi_match_source"
    )
)

new_count = new_events.count()
print(f"\n✓ Enriched {new_count:,} events with gold schema")
print(f"  Matched to assets: {new_events.filter(F.col('asset_id').isNotNull()).count():,}")
new_events.show(5, truncate=False)

stats = new_events.agg(
    F.countDistinct("Tag").alias("unique_tags"),
    F.min("Timestamp").alias("min_ts"),
    F.max("Timestamp").alias("max_ts")
).collect()[0]
print(f"  Tags: {stats['unique_tags']}, Range: {stats['min_ts']} → {stats['max_ts']}")


## Step 3: Dedup and Append to gold.fact_pi
Append new events to the Delta table. Use `dropDuplicates` as a safety net
against overlapping watermark windows.


In [ ]:
# Dedup within the batch (same Tag+Timestamp = keep one)
deduped = new_events.dropDuplicates(["Tag", "Timestamp"])
deduped_count = deduped.count()

if deduped_count < new_count:
    print(f"Deduped: {new_count:,} → {deduped_count:,} ({new_count - deduped_count:,} duplicates removed)")
else:
    print(f"No duplicates found in batch ({deduped_count:,} rows)")

# Append to gold.fact_pi
deduped.write.mode("append").format("delta").saveAsTable(GOLD_TABLE)

# Verify
new_max = spark.sql(f"SELECT MAX(Timestamp) as max_ts, COUNT(*) as total FROM {GOLD_TABLE}").collect()[0]
print(f"\n✓ Appended {deduped_count:,} rows to {GOLD_TABLE}")
print(f"  New watermark: {new_max['max_ts']}")
print(f"  Total rows in {GOLD_TABLE}: {new_max['total']:,}")


## Step 4: Validation
Quick checks to confirm data integrity after the load.


In [ ]:
# Validation: check per-asset coverage after load
print("Post-load validation:")
print("="*70)

# Recent data by tag (last 24h)
recent = spark.sql(f"""
    SELECT 
        b.asset_id,
        COUNT(DISTINCT p.Tag) as active_tags,
        COUNT(*) as readings_24h,
        MIN(p.Timestamp) as oldest,
        MAX(p.Timestamp) as newest
    FROM {GOLD_TABLE} p
    JOIN gold.bridge_pi_tag_to_asset b ON p.Tag = b.Tag
    WHERE p.Timestamp >= current_timestamp() - INTERVAL 24 HOURS
    GROUP BY b.asset_id
    ORDER BY b.asset_id
""")

if recent.count() > 0:
    print("\nLast 24h coverage by asset:")
    recent.show(truncate=False)
else:
    print("\n⚠️ No data in last 24h — check Eventhouse feed")

# Overall table stats
overall = spark.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT Tag) as unique_tags,
        MIN(Timestamp) as earliest,
        MAX(Timestamp) as latest,
        DATEDIFF(MAX(Timestamp), MIN(Timestamp)) as days_span
    FROM {GOLD_TABLE}
""").collect()[0]

print(f"\nOverall {GOLD_TABLE}:")
print(f"  Total rows: {overall['total_rows']:,}")
print(f"  Unique tags: {overall['unique_tags']}")
print(f"  Range: {overall['earliest']} → {overall['latest']} ({overall['days_span']} days)")
print(f"\n✓ PI-Gold-Delta load complete")
